# DataMend: precipitation QC pipeline on Kaggle

Runs the whole rainfall QC system in `ml-service/qc/` (spec: `docs/qc-system-spec.md`) on RF25 gridded daily rainfall:
data check and splits, frozen fault benchmark, untrained baselines, trained components, federated variants, fusion,
budget choice, the single test-scoring pass, and the paper tables, figures and claim check.
(`datamend_training.ipynb` is for the older `training` package, not this system.)

**Before you run it**
1. Notebook settings: **Accelerator** = GPU (T4 x2 is what the `kaggle` profile expects; one GPU or none also works, slower), **Internet** = On (needed to clone the repo and install packages).
2. **Add Input**: attach the RF25 dataset by pasting its link. Then paste the same link into `DATASET_LINK` in the next cell. The dataset must contain every year in the configured ranges (1901-2023 = 123 files).
3. The GitHub repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (**Add-ons, Secrets, Add secret**, then switch it **on** for this notebook). Use a GitHub personal access token with read access to this repository (fine-grained: *Contents: read*, plus *write* if you use `PUSH_RESULTS`; or a classic token with `repo` scope). Create it at github.com, Settings, Developer settings. The token is never printed and never saved in the output.
4. Leave `SMOKE = True` for a first run: it trains on 3 years with a 15 minute cap, under its own run name, and refuses to score the test years. Check that every step works and how long it takes, then set `SMOKE = False`.

**Two phases, on purpose.** Phase 1 runs through the fusion step and then stops: the two false-alarm budgets are *your* decision, made from the validation curves, and must be fixed before any test year is scored. Choose them, fill `BUDGET_FINAL` and `BUDGET_REVIEW`, tick `CONFIRM_TEST_SCORE`, and run again (restore the earlier outputs with `RESUME_LINK`, see below). The test years can be scored **once**.

**Git does the coordination.** Every run starts by fetching the latest commit of `BRANCH` from the repository, so the notebook always runs the code you last pushed (push first). With `PUSH_RESULTS = True` the small results (reports, metrics, tables, figures, the budget log) are committed to a separate branch, `kaggle-results`, after every step (so a lost session keeps them), under `docs/results/<run>/`, never to the code branch and never with a force push. Pull them on your machine with `git fetch origin kaggle-results`. The data itself cannot go through git; it comes from the Kaggle dataset.

**Two GPUs.** With two or more GPUs, `train` and `federated` run as one process per GPU: the (component, seed) units are split between them with `--shard`, each process on its own device. The other steps (prepare, benchmark, untrained, fusion) are CPU work.

**Sessions.** A Kaggle session ends after about 12 hours. Training and the federated runs checkpoint every year and resume. To continue in a new session: *Save Version, Save & Run All* so `/kaggle/working` is kept as the version's output, then in the new session attach that output as an input and put its link in `RESUME_LINK`.

**Status.** This pipeline has been tested on small generated data and on the real RF25 files only as far as the data-check and split step. It has not been run on Kaggle, and no real result exists yet. If a step fails, the error is printed in full; send it along.

## 1. Settings

In [ ]:
# ------------------------------- edit these -------------------------------
DATASET_LINK = "https://www.kaggle.com/datasets/<owner>/<dataset-name>"   # the RF25 dataset: paste its Kaggle link
REPO_URL = "https://github.com/NL0711/datamend.git"
BRANCH = "main"

SMOKE = True                  # True: tiny run under the run name "smoke" (3 train years, 15 min caps, one seed, no test scoring)
RUN_NAME = "base"             # run folder name for the real run (the smoke run always uses "smoke")
SEEDS = None                  # None = the seeds in the config ([0, 1, 2]); or e.g. [0]
TIME_CAP_HOURS = None         # per training command; None = the profile's caps
FEDERATED_VARIANTS = ["fedpers", "fedavg", "fedft", "local"]    # priority order; [] to skip the federated runs
FUSION_ON_FEDERATED = ["fedpers"]                               # also fit a fusion on these federated components; [] to skip
EXTRA_SET = []                # config overrides, e.g. ["years.test=[2018, 2024]"] if you add RF25 2024 (before the first run only)
RESUME_LINK = ""              # link of a previous version's output (attached as input) to continue from

# Phase 2 (after you have looked at the validation curves):
BUDGET_FINAL = None           # false alerts per 1,000 clean cell-days for the final flag, e.g. 1
BUDGET_REVIEW = None          # the same for the review flag (larger), e.g. 5
CONFIRM_TEST_SCORE = False    # True = "I understand the test years are scored once, with these budgets"

STEPS = {"prepare": True, "benchmark": True, "untrained": True, "train": True, "federated": True, "fusion": True,
         "budgets": True, "test_score": True, "paper": True}
PUSH_RESULTS = False          # True: commit the small results to the git branch RESULTS_BRANCH (needs the GITHUB_TOKEN secret with write access)
RESULTS_BRANCH = "kaggle-results"
FULL_ZIP = False              # also zip all outputs (checkpoints and scores can be large); the small results zip is always made

## 2. Get the code and install

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time

SECRET = ""        # the GitHub token, if any: replaced by *** in everything printed

def auth_url(url, token):
    """The repository URL with the token in it, for one command. Never printed (see sh) and never stored in .git/config."""
    return url.replace("https://", f"https://x-access-token:{token}@") if token else url

def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    clean = lambda t: t.replace(SECRET, "***") if SECRET else t
    out = clean((r.stdout + r.stderr).strip())
    if show: print("$", clean(cmd), "\n" + out[-1500:])
    if r.returncode: raise RuntimeError(f"failed: {clean(cmd)[:120]}\n{out[-1500:]}")
    return out

WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else pathlib.Path.cwd()
here = pathlib.Path.cwd()
local = next((p for p in [here, *here.parents] if (p / "ml-service" / "qc").is_dir()), None)   # already inside the repo (local run)
if local:
    REPO = local
else:
    REPO = WORK / "datamend"
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("No GITHUB_TOKEN secret available:", type(e).__name__, "(fine for a public repository)")
    SECRET = token
    os.environ["GIT_TERMINAL_PROMPT"] = "0"          # fail at once instead of waiting for a password that can never be typed
    url = auth_url(REPO_URL, token)
    try:
        if (REPO / ".git").is_dir():
            sh(f"git -C {REPO} fetch {url} {BRANCH} && git -C {REPO} checkout -B {BRANCH} FETCH_HEAD")
        else:
            sh(f"git clone --branch {BRANCH} {url} {REPO}")
            sh(f"git -C {REPO} remote set-url origin {REPO_URL}", show=False)      # keep the token out of .git/config (and the saved output)
    except RuntimeError as e:
        if not token:
            raise RuntimeError(
                "Could not read the repository, and no GitHub token was found. If the repository is private (it is by default), "
                "add a Kaggle secret named GITHUB_TOKEN: Add-ons > Secrets > Add secret, then switch it ON for this notebook. "
                "Use a GitHub personal access token with read access to the repository (Contents: read; add write if you set PUSH_RESULTS). "
                "Then run this cell again.") from None
        raise RuntimeError("Could not read the repository with the GITHUB_TOKEN secret. Check that the token is attached to this notebook, has not "
                           "expired, and has access to " + REPO_URL + ". Original error:\n" + str(e)) from None
    sh(f"git -C {REPO} log -1 --format='%h %s'")
ML = REPO / "ml-service"
os.chdir(ML)
sys.path.insert(0, str(ML))
# torch is preinstalled on Kaggle. The ERA5 extras (cfgrib, eccodes, cdsapi) are not needed for the rainfall pipeline.
sh("pip install -q numpy pandas scipy scikit-learn xarray netCDF4 matplotlib pyyaml pyarrow kagglehub", show=False)
print("repo:", REPO, "| cwd:", os.getcwd())

## 3. Find the RF25 data from the link
Works out where Kaggle mounted the dataset you linked and checks that all years are there. If the dataset is not attached and Internet is on, it falls back to downloading it with `kagglehub`.

In [ ]:
import torch
from qc.config import load_config, year_range
from qc.kaggle_data import locate_rf25, DatasetNotFound

cfg0 = load_config(ML / "qc" / "configs" / "base.yaml", list(EXTRA_SET))
need = sum(len(year_range(cfg0.years[k])) for k in ("train", "val", "test", "stress") if cfg0.years.get(k))
def _download(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)
RF25_DIR = locate_rf25(DATASET_LINK, expected_files=need, download=_download)
print(f"RF25 folder: {RF25_DIR} ({len(list(RF25_DIR.glob('RF25_ind*_rfp25.nc')))} files, {need} needed for the configured years)")

n_gpu = torch.cuda.device_count()
DEVICE_NAMES = [f"cuda:{i}" for i in range(n_gpu)]       # train and federated run one process per GPU when there are two or more
PROFILE = "kaggle" if n_gpu >= 2 else "single-gpu"      # single-gpu also runs on CPU, slowly
print(f"torch {torch.__version__} | GPUs: {n_gpu} {[torch.cuda.get_device_name(i) for i in range(n_gpu)]} | profile: {PROFILE}")
if n_gpu == 0:
    print("WARNING: no GPU. Turn on a GPU in the notebook settings, or the full run will take far too long.")

RUN = "smoke" if SMOKE else RUN_NAME
os.environ["DATAMEND_RF25_DIR"] = str(RF25_DIR)
os.environ["DATAMEND_DATA_DIR"] = str(WORK / "data")       # splits, benchmark, cache: kept in the output
OUT_ROOT = WORK / "artifacts"                                # run folders: <OUT_ROOT>/qc/<RUN>
print("run name:", RUN, "| outputs:", OUT_ROOT / "qc" / RUN)

## 4. Continue from a previous session (optional)

In [ ]:
from qc.kaggle_data import find_saved_outputs
if RESUME_LINK.strip():
    prev = find_saved_outputs(RESUME_LINK)
    if prev is None:
        raise RuntimeError(f"no previous output (a folder with artifacts/qc) found for {RESUME_LINK!r}. Attach the previous version's output as an input first.")
    for sub in ("artifacts", "data"):
        if (prev / sub).is_dir():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print("restored from", prev)
else:
    print("fresh run (no RESUME_LINK)")

## 5. Run helper
Each step runs `python -m qc <step>` with the chosen profile and run name, and streams its output. Steps that were already done resume or skip.

In [ ]:
import subprocess, re
from IPython.display import Image, Markdown, display
from qc.parallel import run_parallel
from qc.benchmark import bench_dir

SEED_ARGS = (["--seeds"] + [str(s) for s in ([0] if SMOKE and SEEDS is None else SEEDS)]) if (SMOKE or SEEDS) else []
SMOKE_ARGS = ["--max-train-years", "3", "--time-cap-hours", "0.25"] if SMOKE else (["--time-cap-hours", str(TIME_CAP_HOURS)] if TIME_CAP_HOURS else [])
COMMON = ["--profile", PROFILE, "--set", f"name={RUN}"] + [x for s in EXTRA_SET for x in ("--set", s)]

def small_files():
    """The small results worth keeping: reports, metrics, tables, figures, curves, budget log inputs, manifests. No checkpoints or scores."""
    run_dir = OUT_ROOT / "qc" / RUN
    keep = []
    for pat in ("test/test_report.*", "test/metrics_*.json", "paper/**/*", "fusion/validation/*", "fusion_budgets.json", "*/meta.json", "*/*.json"):
        keep += [p for p in run_dir.glob(pat) if p.is_file() and p.stat().st_size < 50e6]
    work = pathlib.Path(os.environ["DATAMEND_DATA_DIR"])
    keep += [p for p in list(work.glob("splits/*.json")) + list(bench_dir(cfg).glob("manifest.json")) if p.is_file()]
    return list(dict.fromkeys(keep))

_push_note = []
def push_small(label):
    """With PUSH_RESULTS: commit the small results to RESULTS_BRANCH (built in a temporary worktree: this checkout is not touched).
    Runs after every step so a lost session keeps the metrics and the budget log. A failed push is reported, never fatal."""
    if not PUSH_RESULTS:
        return
    if local or not SECRET:
        if not _push_note:
            print("PUSH_RESULTS skipped:", "running inside a local checkout" if local else "no GITHUB_TOKEN secret (it needs write access)")
            _push_note.append(1)
        return
    try:
        from qc.kaggle_sync import sync_results
        files = {p: str(p.relative_to(WORK)) for p in small_files() if p.is_relative_to(WORK)}
        protocol = REPO / "docs" / "qc-evaluation-protocol.md"
        tracked = {"docs/qc-evaluation-protocol.md": protocol.read_text(encoding="utf-8")} if protocol.is_file() else {}
        sha = sync_results(REPO, auth_url(REPO_URL, SECRET), RUN, files, RESULTS_BRANCH, tracked, message=f"results: {RUN} after {label}", secret=SECRET)
        print(f"[results pushed to {RESULTS_BRANCH}: {sha}]" if sha else f"[nothing new to push to {RESULTS_BRANCH}]")
    except Exception as e:
        print("[results push failed, the run continues:", str(e)[:300], "]")

def qc_gpus(step, *args, seeds=False, capped=False):
    """`qc <step>` once per GPU: each process takes its share of the (component, seed) units (--shard I/N) on its own device."""
    n = len(DEVICE_NAMES)
    if n < 2:
        return qc(step, *args, seeds=seeds, capped=capped)
    cmds = [[sys.executable, "-u", "-m", "qc", step, *args, *(SEED_ARGS if seeds else []), *(SMOKE_ARGS if capped else []),
             "--shard", f"{i}/{n}", *COMMON, "--set", f"compute.devices=[{d}]"] for i, d in enumerate(DEVICE_NAMES)]
    print(f"\n$ ({n} processes, one per GPU)", " ".join(cmds[0][2:]), flush=True)
    t = time.time()
    codes = run_parallel(cmds, cwd=ML, labels=[f"gpu{i}" for i in range(n)])
    print(f"[{step}: exit codes {codes} after {(time.time() - t) / 60:.1f} min]")
    if any(codes):
        raise RuntimeError(f"`qc {step}` failed on {sum(1 for c in codes if c)} of {n} GPUs (exit codes {codes}; 3 = stopped at the time cap, "
                           "run the cell again to resume); the output above has the reason")
    push_small(step)

def qc(step, *args, seeds=False, capped=False):
    cmd = [sys.executable, "-u", "-m", "qc", step, *args, *(SEED_ARGS if seeds else []), *(SMOKE_ARGS if capped else []), *COMMON]
    print("\n$", " ".join(cmd[2:]), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, cwd=ML, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail.append(line)
        tail = tail[-40:]
    rc = p.wait()
    print(f"[{step}: exit {rc} after {(time.time() - t) / 60:.1f} min]")
    if rc:
        raise RuntimeError(f"`qc {step}` failed (exit {rc}); the output above has the reason")
    push_small(step)
    return "".join(tail)

cfg = load_config(ML / "qc" / "configs" / "base.yaml", list(EXTRA_SET) + [f"name={RUN}"])
PHASE2 = False

## 6. Data check, splits and statistics

In [ ]:
if STEPS["prepare"]:
    from qc.prepare import load_prepared
    from qc.grid import ContractError
    try:
        load_prepared(cfg)          # fast: loads the saved split and statistics and checks they match this config, map and land mask
        print("split manifest and training statistics already exist and match this configuration: prepare skipped "
              "(delete data/splits in the output to force it)")
    except ContractError:
        qc("prepare")               # data contract on every year, split manifest, leakage audit, training statistics

## 7. Frozen fault benchmark

In [ ]:
if STEPS["benchmark"]:
    mf = bench_dir(cfg) / "manifest.json"
    split_hash = json.loads((pathlib.Path(os.environ["DATAMEND_DATA_DIR"]) / "splits" / "split_manifest.json").read_text())["hash"]
    if mf.is_file() and json.loads(mf.read_text()).get("split_manifest_hash") == split_hash:
        print("frozen benchmark already exists for this split:", bench_dir(cfg), "(not rebuilt)")
    else:
        if mf.is_file():
            print("the existing benchmark was built for a different split (the year ranges changed): rebuilding it")
        qc("benchmark", *(["--overwrite"] if mf.is_file() else []))

## 8. Untrained baselines (rules, slow component, neighbour average)

In [ ]:
if STEPS["untrained"]:
    qc("untrained")

## 9. Trained components (spatial, temporal, plain GRU)

In [ ]:
if STEPS["train"]:
    qc_gpus("train", seeds=True, capped=True)

## 10. Federated variants

In [ ]:
if STEPS["federated"] and FEDERATED_VARIANTS:
    qc_gpus("federated", "--variants", *FEDERATED_VARIANTS, seeds=True, capped=True)

## 11. Fusion and validation curves
Fits the fusion on synthetic faults of validation years and draws recall against the false-alarm budget. **Phase 1 ends here.**

In [ ]:
if STEPS["fusion"]:
    qc("fusion", seeds=True)
    for src in FUSION_ON_FEDERATED:
        qc("fusion", "--on", src, seeds=True)

In [ ]:
from IPython.display import Image, display
import pandas as pd
from qc.evaluate import summary_row
val = OUT_ROOT / "qc" / RUN / "fusion" / "validation"
seed0 = (SEEDS or [0])[0] if not SMOKE else 0
png = val / f"curves_s{seed0}.png"
if png.is_file():
    display(Image(str(png)))
    comp = json.loads((val / f"comparison_s{seed0}.json").read_text())
    rows = [summary_row(r, "seen", b) for r in comp.values() for b in (1.0,) if str(float(b)) in r["groups"]["seen"]["by_budget"]]
    print("Validation (synthetic faults), false-alarm budget 1 per 1,000 clean cell-days:")
    display(pd.DataFrame(rows).round(3))
else:
    print("no validation curves yet:", png)

## 12. Phase 2: fix the budgets
Choose the **final** and **review** false-alarm budgets (false alerts per 1,000 clean cell-days; review larger than final) from the curves above, set `BUDGET_FINAL` and `BUDGET_REVIEW` in the settings, and run from here. They are written to a log and checked before any test scoring; they cannot be changed afterwards. Not set: this cell does nothing and the notebook ends cleanly after phase 1.

In [ ]:
if STEPS["budgets"]:
    if BUDGET_FINAL is None or BUDGET_REVIEW is None:
        print("Budgets not set. Look at the validation curves above, set BUDGET_FINAL and BUDGET_REVIEW in section 1, then run from section 1 again (with RESUME_LINK if this is a new session).")
    else:
        qc("set-budgets", "--final", str(BUDGET_FINAL), "--review", str(BUDGET_REVIEW), "--seed", "0")
        PHASE2 = True

## 13. Test scoring (single use)
Scores the test years 2018-2023 once with the fixed budgets, then the unseen clients and the 1901-1950 stress period. Needs `CONFIRM_TEST_SCORE = True`. Refused in smoke mode, and refused a second time by the pipeline itself.

In [ ]:
if STEPS["test_score"] and PHASE2:
    if SMOKE:
        print("Smoke run: the test years are not scored. Set SMOKE = False for the real run.")
    elif not CONFIRM_TEST_SCORE:
        print("Set CONFIRM_TEST_SCORE = True to score the test years. This can be done once.")
    else:
        qc("test-score", seeds=True)
        report = OUT_ROOT / "qc" / RUN / "test" / "test_report.md"
        from IPython.display import Markdown
        display(Markdown(report.read_text(encoding="utf-8")))

## 14. Paper tables, figures and claim check

In [ ]:
if STEPS["paper"] and PHASE2 and not SMOKE:
    qc("paper")
    from IPython.display import Markdown
    paper = OUT_ROOT / "qc" / RUN / "paper"
    for f in (paper / "claims.md", paper / "tables" / "all_tables.md"):
        if f.is_file():
            display(Markdown(f.read_text(encoding="utf-8")))
    for png in sorted((paper / "figures").glob("*.png")):
        print(png.name); display(Image(str(png)))

## 15. Package the results
`qc_results_small.zip` holds what is worth keeping in git or sending back: reports, metrics, tables, figures, validation curves, the budget log, the split and benchmark manifests. It leaves out checkpoints, score files and alert records. Everything is also kept in the notebook output (`/kaggle/working`) if you save a version.

In [ ]:
import zipfile
keep = small_files()
protocol = REPO / "docs" / "qc-evaluation-protocol.md"
small = WORK / f"qc_results_small_{RUN}.zip"
with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as z:
    for p in dict.fromkeys(keep):
        z.write(p, p.relative_to(WORK) if p.is_relative_to(WORK) else p.name)
    if protocol.is_file():
        z.write(protocol, "docs/qc-evaluation-protocol.md")
print(f"{small} ({small.stat().st_size / 1e6:.1f} MB, {len(set(keep))} files)")
push_small("final")
if FULL_ZIP:
    full = shutil.make_archive(str(WORK / f"qc_all_outputs_{RUN}"), "zip", WORK, "artifacts")
    print("full:", full, f"({os.path.getsize(full) / 1e9:.2f} GB)")
print("\nSummary: run", RUN, "| smoke" if SMOKE else "| real", "| budgets", (BUDGET_FINAL, BUDGET_REVIEW),
      "| phase 2 done" if PHASE2 else "| phase 1 only")